# 03 - Erro com pontos flutuantes
Vamos aprender sobre como usar os erros de ponto flutuante para resolução de problemas numéricos.

Crie uma nova branch (versão) do repositório:

```bash
git branch semana3
```

Faça o checkout nessa nova branch:

```bash
git checkout semana3
```

<hr />

## Atividade 1
A função exponencial natural pode ser definida pelo limite:
$$
e^x=\lim_{n\to\infty}\left(1+\frac{x}{n}\right)^n,
$$
mas também é dada pela **série de Maclaurin**:
$$
e^x=\sum_{n=0}^{\infty}\frac{x^n}{n!}
=1+\frac{x}{1!}+\frac{x^2}{2!}+\frac{x^3}{3!}+\cdots
$$

Implemente em **Python** o cálculo de $e^x$ pela série, interrompendo a soma quando o termo ficar menor que o limite prático de contribuição, usando a precisão de máquina como critério.

In [16]:
import math
import sys

def exp_series(x, atol=0.0):
    """ Aproxima e^x pela série de Maclaurin com critério de parada numérico. """
    eps = sys.float_info.epsilon
    s = 1.0
    term = 1.0
    n = 0
    tol_abs = max(atol, eps)

    while True:
        n += 1
        term *= x / n
        s += term
        if abs(term) < eps * abs(s) or abs(term) < tol_abs:
            break
        if n > 10_000:
            break
    return s, n, term

for val in [1.0, 5.0, -2.0]:
    approx, nterms, last = exp_series(val)
    print(f"x={val:+g} -> e^x ≈ {approx:.16g} (math.exp={math.exp(val):.16g}, termos={nterms})")

x=+1 -> e^x ≈ 2.718281828459046 (math.exp=2.718281828459045, termos=18)
x=+5 -> e^x ≈ 148.4131591025766 (math.exp=148.4131591025766, termos=33)
x=-2 -> e^x ≈ 0.1353352832366127 (math.exp=0.1353352832366127, termos=24)


## Atividade 2

Implemente:
$$
e^x\approx\left(1+\frac{x}{n}\right)^n
$$
com $n$ crescente, e:
1. Explique por que, para $x<0$ e $n$ muito grande, pode ocorrer **cancelamento catastrófico**;
2. Proponha um critério de parada numérico para encerrar o crescimento de $n$ sem perder precisão.

In [28]:
import math

def aproxima_exp(x, tol=1e-15):
    n = 1
    a = (1 + x/n)**n

    while True:
        n *= 2
        base = 1 + x/n

        if base == 1.0:
            return a, n

        novo = base**n
        erro = abs(novo - a) / max(1.0, abs(novo))

        if erro < tol:
            return novo, n

        a = novo


valores = [-20, -10, -1, 1, 10, 20]

for x in valores:
    resultado, n = aproxima_exp(x)

    valor_exato = math.exp(x)
    erro_relativo = abs(resultado - valor_exato) / abs(valor_exato)

    print(f"x = {x}")
    print(f"n = {n}")
    print(f"aproximação = {resultado:.15e}")
    print(f"math.exp(x) = {valor_exato:.15e}")
    print(f"erro relativo = {erro_relativo:.5e}")
    print()

x = -20
n = 536870912
aproximação = 2.061152854599122e-09
math.exp(x) = 2.061153622438558e-09
erro relativo = 3.72529e-07

x = -10
n = 4398046511104
aproximação = 4.539992976196871e-05
math.exp(x) = 4.539992976248485e-05
erro relativo = 1.13688e-11

x = -1
n = 281474976710656
aproximação = 3.678794411714417e-01
math.exp(x) = 3.678794411714423e-01
erro relativo = 1.81074e-15

x = 1
n = 562949953421312
aproximação = 2.718281828459043e+00
math.exp(x) = 2.718281828459045e+00
erro relativo = 8.16856e-16

x = 10
n = 36028797018963968
aproximação = 2.980957987041726e+03
math.exp(x) = 2.202646579480672e+04
erro relativo = 8.64665e-01

x = 20
n = 288230376151711744
aproximação = 7.896296018268042e+13
math.exp(x) = 4.851651954097903e+08
erro relativo = 1.62754e+05



## Atividade 3

Para $|x|$ grande, use:
$$
e^x = \left(e^{m\cdot 2^{-k}}\right)^{2^k}, \quad
k = \left\lceil \log_2\!\left(\frac{|x|}{\theta}\right)\right\rceil, \quad m = \frac{x}{2^k}
$$
Calcule $e^{m}$ pela série (Ex. 1) e depois eleve ao quadrado $k$ vezes.

In [29]:
import math

def exp_reduzido(x, theta=1.0, tol=1e-15):
    if x == 0:
        return 1.0

    k = math.ceil(math.log2(abs(x) / theta)) if abs(x) > theta else 0
    m = x / (2**k)

    termo = 1.0
    soma = 1.0
    j = 1

    while abs(termo) > tol:
        termo *= m / j
        soma += termo
        j += 1

    resultado = soma

    for _ in range(k):
        resultado *= resultado

    return resultado


valores = [1, 5, 10, 20, 40, 50]

for x in valores:
    resultado = exp_reduzido(x)
    valor_exato = math.exp(x)
    erro_relativo = abs(resultado - valor_exato) / abs(valor_exato)

    print(f"x = {x}")
    print(f"e^x aproximado = {resultado:.15e}")
    print(f"math.exp(x)     = {valor_exato:.15e}")
    print(f"erro relativo   = {erro_relativo:.5e}")
    print()

x = 1
e^x aproximado = 2.718281828459046e+00
math.exp(x)     = 2.718281828459045e+00
erro relativo   = 1.63371e-16

x = 5
e^x aproximado = 1.484131591025767e+02
math.exp(x)     = 1.484131591025766e+02
erro relativo   = 5.74512e-16

x = 10
e^x aproximado = 2.202646579480674e+04
math.exp(x)     = 2.202646579480672e+04
erro relativo   = 9.90984e-16

x = 20
e^x aproximado = 4.851651954097913e+08
math.exp(x)     = 4.851651954097903e+08
erro relativo   = 2.08852e-15

x = 40
e^x aproximado = 2.353852668370210e+17
math.exp(x)     = 2.353852668370200e+17
erro relativo   = 4.07842e-15

x = 50
e^x aproximado = 5.184705528587007e+21
math.exp(x)     = 5.184705528587072e+21
erro relativo   = 1.25391e-14



## Atividade 4

Use:
$$
\cos x=\sum_{n=0}^{\infty}(-1)^n\frac{x^{2n}}{(2n)!}
$$
com a recursão:
$$
t_{n+1}=t_n\cdot\frac{-x^2}{(2n+1)(2n+2)}
$$
Defina um critério de parada baseado em `epsilon` e compare o erro relativo para $x\in[-20,20]$ (200 pontos) contra `math.cos(x)`.

In [30]:
import math

def cos_aprox(x, epsilon=1e-15):
    t = 1.0
    soma = t
    n = 0

    while True:
        t = t * (-x**2) / ((2*n + 1)*(2*n + 2))
        soma_novo = soma + t

        if abs(t) <= epsilon * max(1.0, abs(soma_novo)):
            return soma_novo

        soma = soma_novo
        n += 1

x_values = [-20 + i * 40 / 199 for i in range(200)]

erros_relativos = []

for x in x_values:
    valor_aprox = cos_aprox(x)
    valor_exato = math.cos(x)

    if valor_exato != 0:
        erro_relativo = abs(valor_aprox - valor_exato) / abs(valor_exato)
    else:
        erro_relativo = abs(valor_aprox - valor_exato)

    erros_relativos.append(erro_relativo)

print("Erro relativo máximo:", max(erros_relativos))
print("Erro relativo médio:", sum(erros_relativos) / len(erros_relativos))

for i in range(0, 200, 20):
    print(
        "x =", x_values[i],
        "cos_aprox =", cos_aprox(x_values[i]),
        "math.cos =", math.cos(x_values[i]),
        "erro relativo =", erros_relativos[i]
    )

Erro relativo máximo: 8.229597998835908e-09
Erro relativo médio: 2.179804939397409e-10
x = -20.0 cos_aprox = 0.40808205845504064 math.cos = 0.40808206181339196 erro relativo = 8.229597998835908e-09
x = -15.979899497487438 cos_aprox = -0.963252636771032 math.cos = -0.963252636821362 erro relativo = 5.225001260938254e-11
x = -11.959798994974875 cos_aprox = 0.821607204713081 math.cos = 0.8216072047115193 erro relativo = 1.9008483856815325e-12
x = -7.939698492462311 cos_aprox = -0.08561193132275005 math.cos = -0.08561193132272306 erro relativo = 3.1528662966904523e-13
x = -3.91959798994975 cos_aprox = -0.7123149287189787 math.cos = -0.712314928718979 erro relativo = 3.117225204367883e-16
x = 0.10050251256281584 cos_aprox = 0.9949538721054193 math.cos = 0.9949538721054193 erro relativo = 0.0
x = 4.120603015075378 cos_aprox = -0.5578441660464476 math.cos = -0.5578441660464474 erro relativo = 3.980405612174913e-16
x = 8.14070351758794 cos_aprox = -0.28280945935353746 math.cos = -0.28280945935

## Atividade 5

Dado $x$ e uma tolerância $\tau$, encontre o menor $N$ tal que:
$$
R_{N+1}(x)=\sum_{n=N+1}^{\infty}\frac{|x|^n}{n!} < \tau
$$

In [31]:
import math

def menor_N(x, tau):
    termo = 1.0
    soma = 0.0
    n = 0

    while True:
        soma += termo

        if n >= 1:
            resto = abs(x)**(n + 1) / math.factorial(n + 1)

            if resto * (1 + abs(x) / (n + 2)) < tau:
                return n

        n += 1
        termo *= abs(x) / n


valores = [1, 5, 10, 20]
tau = 1e-12

for x in valores:
    N = menor_N(x, tau)
    print(f"x = {x}, tau = {tau}, N = {N}")

x = 1, tau = 1e-12, N = 14
x = 5, tau = 1e-12, N = 30
x = 10, tau = 1e-12, N = 46
x = 20, tau = 1e-12, N = 75


## Atividade 6

Usando `decimal` ou `mpmath`, compute $e^x$ em alta precisão e compare com o resultado de `float64` (Ex. 1) para $x\in\{20, 40, 50\}$.
Analise:
- perda de dígitos significativos;
- quando o `float64` começa a saturar por overflow.


In [32]:
import math
import sys
from decimal import Decimal, getcontext

getcontext().prec = 80

def exp_float64(x, epsilon=1e-15):
    termo = 1.0
    soma = 1.0
    n = 1

    while True:
        termo *= x / n
        novo = soma + termo

        if abs(termo) <= epsilon * abs(novo):
            return novo

        soma = novo
        n += 1

def exp_alta_precisao(x):
    x = Decimal(x)
    termo = Decimal(1)
    soma = Decimal(1)
    n = 1

    while True:
        termo *= x / Decimal(n)
        novo = soma + termo

        if abs(termo) < Decimal("1e-75"):
            return novo

        soma = novo
        n += 1

valores = [20, 40, 50]

print(
    f"{'x':>5} "
    f"{'float64':>25} "
    f"{'alta precisão':>30} "
    f"{'erro relativo':>20} "
    f"{'dígitos':>12}"
)

for x in valores:
    valor_float = exp_float64(x)
    valor_decimal = exp_alta_precisao(x)

    erro = abs(
        Decimal(str(valor_float)) - valor_decimal
    ) / abs(valor_decimal)

    digitos = -math.log10(float(erro)) if erro != 0 else float("inf")

    print(
        f"{x:5d} "
        f"{valor_float:25.16e} "
        f"{str(valor_decimal):>30} "
        f"{float(erro):20.8e} "
        f"{digitos:12.6f}"
    )

limite_overflow = math.log(sys.float_info.max)

print("\nLimite de overflow do float64:")
print(limite_overflow)

print("\nTeste de overflow:")

for x in [700, 709, 709.78, 710, 720]:
    try:
        print(x, math.exp(x))
    except OverflowError:
        print(x, "overflow")

    x                   float64                  alta precisão        erro relativo      dígitos
   20    4.8516519540979028e+08 485165195.40979027796910683054154055868463898894484725435361080031597799614270975       4.54090553e-17    16.342858
   40    2.3538526683701997e+17 235385266837019985.40789991074903480450887161725455546723665125118928916352581697       6.54582172e-17    16.184036
   50    5.1847055285870794e+21 5184705528587072464087.4533229334853848274691005838464019040569338068568847937965       1.26061403e-15    14.899418

Limite de overflow do float64:
709.782712893384

Teste de overflow:
700 1.0142320547350045e+304
709 8.218407461554972e+307
709.78 1.7928227943945155e+308
710 overflow
720 overflow


## Versionando o código

Submeta a branch para o servidor:

```bash
git add .
git commit -m "Semana 3"
git push origin semana3
```